In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.2 Pre-normalization

PocketLM normalizes *before* each sublayer (`attn(norm(x))`), not after. Pre-norm
keeps the residual stream clean and is what lets modern transformers train deep
without careful warmup tricks.

In [ ]:
from torch import nn

torch.manual_seed(0)
dim = 8
ln = nn.LayerNorm(dim)
x = torch.randn(1, 4, dim) * 5 + 3   # arbitrary scale and shift
normed = ln(x)
row = normed[0, 0]
print("each position is normalized: mean", round(row.mean().item(), 6),
      "std", round(row.std(unbiased=False).item(), 4))

LayerNorm rescales each position vector to mean 0, unit variance (then a learned
affine). That stable input is what each sublayer sees.

In [ ]:
assert abs(row.mean().item()) < 1e-5
assert abs(row.std(unbiased=False).item() - 1.0) < 1e-4